In [1]:
!pip install -q -U transformers datasets accelerate peft bitsandbytes trl sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 65.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 527.0/527.0 kB 16.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 680.7/680.7 kB 13.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 7.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 697.4/697.4 kB 24.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 11.4 MB/s eta 0:00:00


In [2]:
import torch
from datasets import load_dataset
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig, pipeline


from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
from trl import SFTTrainer, SFTConfig

In [3]:
dataset = load_dataset("mlabonne/guanaco-llama2-1k", split="train")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001-9ad84bb9cf65a4(…):   0%|          | 0.00/967k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1000 [00:00<?, ? examples/s]

In [4]:
model_name = "NousResearch/Llama-2-7b-chat-hf"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
    bnb_4bit_compute_dtype=torch.float16,
)

tokenizer = AutoTokenizer.from_pretrained(model_name, use_fast=True)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    model_name,
    device_map="auto",
    quantization_config=bnb_config,
)

model = prepare_model_for_kbit_training(model)

config.json:   0%|          | 0.00/583 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/746 [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/500k [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/21.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/435 [00:00<?, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/200 [00:00<?, ?B/s]

In [5]:
lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"]
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

trainable params: 16,777,216 || all params: 6,755,192,832 || trainable%: 0.2484


In [6]:
training_args = SFTConfig(
    output_dir="./llama2-guanaco-lora",

    # =====================
    # Training core
    # =====================
    num_train_epochs=1,
    per_device_train_batch_size=2,          # safer for T4 than 4
    gradient_accumulation_steps=4,          # effective batch = 8
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_ratio=0.03,

    max_grad_norm=0.5,

    # =====================
    # Optimization (QLoRA friendly)
    # =====================
    optim="paged_adamw_8bit",              # best for 4-bit training
    weight_decay=0.001,

    # =====================
    # Precision (IMPORTANT for T4)
    # =====================
    fp16=False,
    bf16=False,

    # =====================
    # Logging / monitoring
    # =====================
    logging_steps=10,
    report_to="tensorboard",

    # =====================
    # Saving
    # =====================
    save_strategy="steps",
    save_steps=200,
    save_total_limit=2,

    # =====================
    # Dataset handling
    # =====================
    dataset_text_field="text",
    max_length=512,
    truncation_mode="keep_start",

    packing=False,

    # =====================
    # Stability / performance
    # =====================
    gradient_checkpointing=True,
    dataloader_num_workers=2,

    # =====================
    # Safety defaults
    # =====================
    remove_unused_columns=True,
)

[transformers] warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [7]:
trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=dataset,
    processing_class=tokenizer
)

Adding EOS to train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

Tokenizing train dataset:   0%|          | 0/1000 [00:00<?, ? examples/s]

sample:

prompt: Write me a short paragraph about the advantages and disadvantages of having daily cold showers.

Answer: Some of the advantages include increased alertness and energy levels, improved circulation, and reduced muscle soreness. Cold showers are also said to boost the immune system, reduce stress levels, and improve skin and hair health. On the other hand, some of the disadvantages include a decrease in body temperature which can lead to decreased physical performance, skin dryness and discomfort, and discomfort during the shower itself. Additionally, for individuals with certain medical conditions such as Raynaud's disease, cold showers can be dangerous and should be avoided. It's important to weigh the potential benefits and drawbacks before incorporating daily cold showers into your routine.

In [8]:
# ======================
# Test sample prompt on untrained model.
# ======================

pipe = pipeline(
    "text-generation",
    model=trainer.model,
    tokenizer=tokenizer,
    device_map="auto"
)

prompt = "<s>[INST] Write me a short paragraph about the advantages and disadvantages of having daily cold showers. [/INST]"

out = pipe(prompt, max_new_tokens=200)
print(out[0]["generated_text"])

[transformers] Passing `generation_config` together with generation-related arguments=({'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s>[INST] Write me a short paragraph about the advantages and disadvantages of having daily cold showers. [/INST]  Taking daily cold showers can have both advantages and disadvantages. One of the main advantages is improved circulation and immune system function. Cold water causes the blood vessels to constrict, which can help increase circulation and improve the delivery of white blood cells to areas of the body that need it. Additionally, cold showers can help boost the immune system by increasing the release of antioxidants and anti-inflammatory compounds. Another advantage of taking daily cold showers is improved mental clarity and alertness. Cold water can increase the release of certain neurotransmitters, such as norepinephrine, which can help improve focus and concentration. However, there are also some disadvantages to taking daily cold showers. One of the main disadvantages is the potential for hypothermia, especially in colder climates or for people with sensitive bodies. Col

In [9]:
trainer.train()

[transformers] The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 2}.


Step,Training Loss
10,1.687329
20,1.395903
30,1.339711
40,1.324375
50,1.395694
60,1.272412
70,1.257006
80,1.224654
90,1.221026
100,1.308388


TrainOutput(global_step=125, training_loss=1.317243766784668, metrics={'train_runtime': 1705.6847, 'train_samples_per_second': 0.586, 'train_steps_per_second': 0.073, 'total_flos': 1.723609488728064e+16, 'train_loss': 1.317243766784668})

In [11]:
model = trainer.model
model.eval()


PeftModelForCausalLM(
  (base_model): LoraModel(
    (model): LlamaForCausalLM(
      (model): LlamaModel(
        (embed_tokens): Embedding(32000, 4096, padding_idx=0)
        (layers): ModuleList(
          (0-31): 32 x LlamaDecoderLayer(
            (self_attn): LlamaAttention(
              (q_proj): lora.Linear4bit(
                (base_layer): Linear4bit(in_features=4096, out_features=4096, bias=False)
                (lora_dropout): ModuleDict(
                  (default): Dropout(p=0.05, inplace=False)
                )
                (lora_A): ModuleDict(
                  (default): Linear(in_features=4096, out_features=16, bias=False)
                )
                (lora_B): ModuleDict(
                  (default): Linear(in_features=16, out_features=4096, bias=False)
                )
                (lora_embedding_A): ParameterDict()
                (lora_embedding_B): ParameterDict()
                (lora_magnitude_vector): ModuleDict()
              )
             

In [13]:
# ======================
# Test sample prompt on trained model.
# ======================
pipe = pipeline(
    "text-generation",
    model=model,
    tokenizer=tokenizer
)

prompt = "<s>[INST] Write me a short paragraph about the advantages and disadvantages of having daily cold showers. [/INST]"

out = pipe(prompt, max_new_tokens=200)

print(out[0]["generated_text"])

[transformers] Both `max_new_tokens` (=200) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


<s>[INST] Write me a short paragraph about the advantages and disadvantages of having daily cold showers. [/INST] Cold showers can have both positive and negative effects on the body. On the one hand, they can increase circulation, boost the immune system, and improve skin health. On the other hand, they can also be uncomfortable and even painful, especially in colder climates. Additionally, cold showers can make the skin more sensitive to the cold, which can lead to hypothermia in extreme cases. Overall, the benefits of cold showers are generally considered to outweigh the drawbacks, but it is important to use caution and to gradually acclimate oneself to the cold water. 
